# See My Voice: Mandarin Pronunciation Prototype

This notebook is a beginner-friendly prototype for a Mandarin pronunciation training app.

It starts with a practical MVP:

1. Convert Chinese text to pinyin with tones.
2. Split pinyin into initial, final, and tone.
3. Load a user recording.
4. Extract pitch/F0 and score Mandarin tones.
5. Optionally run a phoneme CTC model.
6. Prepare a JSON response that a web/mobile app can visualize.

Important: the optional CTC model in this notebook is a generic multilingual phoneme recognizer, not a Mandarin-specialized pinyin model. It is useful for learning the pipeline. A real product should later fine-tune a Mandarin pinyin/phone CTC model.

## 0. Install Dependencies

Run this cell once if your environment does not have the packages yet. The `transformers`, `torch`, and `torchaudio` packages are only needed for the optional CTC section.

In [ ]:
# Run once if imports fail:
# %pip install -r requirements.txt

# Or install directly:
# %pip install pypinyin numpy scipy librosa soundfile matplotlib transformers torch torchaudio jupyter

## 1. Imports And Project Settings

Put one test recording in this folder and name it `sample.wav`. The recording should contain the target phrase only, for example saying `你好`.

In [ ]:
from pathlib import Path
import json
import math
import re

import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path(r"C:\Users\jiawen.sun_sx\Desktop\see-my-voice")
AUDIO_PATH = PROJECT_DIR / "sample.wav"

TARGET_TEXT = "你好"
SAMPLE_RATE = 16_000

PROJECT_DIR, AUDIO_PATH.exists()

## 2. Chinese Text To Pinyin

For the first prototype, `pypinyin` is the easiest way to get pinyin with tone numbers.

Example:

`你好` becomes `ni2 hao3` when tone sandhi is enabled. Without tone sandhi, it is often shown as `ni3 hao3`.

In [ ]:
from pypinyin import Style, lazy_pinyin

def text_to_pinyin(text: str, use_tone_sandhi: bool = True):
    return lazy_pinyin(
        text,
        style=Style.TONE3,
        tone_sandhi=use_tone_sandhi,
        neutral_tone_with_five=True,
        errors="ignore",
    )

pinyin_syllables = text_to_pinyin(TARGET_TEXT)
pinyin_syllables

## 3. Split Pinyin Into Initial, Final, Tone

Mandarin syllables are usually easier to score as separate parts:

- initial: consonant at the beginning, like `n`, `h`, `zh`
- final: vowel/nasal part, like `i`, `ao`, `ang`
- tone: `1`, `2`, `3`, `4`, or `5` for neutral tone

In [ ]:
MANDARIN_INITIALS = [
    "zh", "ch", "sh",
    "b", "p", "m", "f", "d", "t", "n", "l",
    "g", "k", "h", "j", "q", "x", "r", "z", "c", "s",
    "y", "w",
]

def split_pinyin_syllable(syllable: str):
    s = syllable.lower().replace("u:", "v")
    match = re.search(r"([1-5])$", s)
    tone = match.group(1) if match else "5"
    base = re.sub(r"[1-5]$", "", s)

    for initial in MANDARIN_INITIALS:
        if base.startswith(initial):
            final = base[len(initial):]
            return {
                "pinyin": syllable,
                "initial": initial,
                "final": final,
                "tone": tone,
            }

    return {
        "pinyin": syllable,
        "initial": "",
        "final": base,
        "tone": tone,
    }

syllable_parts = [split_pinyin_syllable(s) for s in pinyin_syllables]
syllable_parts

## 4. Load User Audio

Save a short recording as:

`C:\Users\jiawen.sun_sx\Desktop\see-my-voice\sample.wav`

For best results at this stage:

- record only the target phrase
- avoid background noise
- leave little silence before and after speech

In [ ]:
import librosa
import soundfile as sf

def load_audio(path: Path, sample_rate: int = SAMPLE_RATE):
    audio, sr = librosa.load(path, sr=sample_rate, mono=True)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    return audio, sr

if AUDIO_PATH.exists():
    audio, sr = load_audio(AUDIO_PATH)
    duration = len(audio) / sr
    print(f"Loaded {AUDIO_PATH}")
    print(f"sample_rate={sr}, duration={duration:.2f}s")
else:
    audio, sr = None, SAMPLE_RATE
    print(f"No audio found yet. Add a recording here: {AUDIO_PATH}")

In [ ]:
def plot_waveform(audio, sr, title="Waveform"):
    times = np.arange(len(audio)) / sr
    plt.figure(figsize=(12, 3))
    plt.plot(times, audio, linewidth=0.8)
    plt.title(title)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.25)
    plt.show()

if audio is not None:
    plot_waveform(audio, sr, title=f"User recording: {TARGET_TEXT}")


## 5. Pitch / F0 Extraction For Tone Feedback

Mandarin tone is mainly about pitch shape over time. We use `librosa.pyin` to estimate F0.

This is not perfect, especially for noisy recordings or atypical voices, but it is a good beginner baseline.

In [ ]:
def estimate_f0(audio, sr, fmin=60, fmax=500):
    f0, voiced_flag, voiced_prob = librosa.pyin(
        audio,
        fmin=fmin,
        fmax=fmax,
        sr=sr,
        frame_length=1024,
        hop_length=160,
    )
    times = librosa.frames_to_time(np.arange(len(f0)), sr=sr, hop_length=160)
    return times, f0, voiced_flag, voiced_prob

if audio is not None:
    f0_times, f0, voiced_flag, voiced_prob = estimate_f0(audio, sr)
    plt.figure(figsize=(12, 3))
    plt.plot(f0_times, f0, marker=".", linewidth=1)
    plt.title("Estimated pitch / F0")
    plt.xlabel("Time (seconds)")
    plt.ylabel("F0 (Hz)")
    plt.grid(True, alpha=0.25)
    plt.show()
else:
    f0_times, f0, voiced_flag, voiced_prob = None, None, None, None

## 6. Naive Syllable Windows

A real system should align syllables using CTC forced alignment or a forced aligner.

For this first notebook, we use a simple beginner approximation: split the speech duration evenly across syllables. This lets us build and test the tone-scoring idea before the aligner exists.

In [ ]:
def naive_syllable_windows(total_duration: float, n_syllables: int):
    edges = np.linspace(0.0, total_duration, n_syllables + 1)
    return [(float(edges[i]), float(edges[i + 1])) for i in range(n_syllables)]

if audio is not None:
    windows = naive_syllable_windows(len(audio) / sr, len(syllable_parts))
else:
    windows = []

list(zip([s["pinyin"] for s in syllable_parts], windows))

## 7. Tone Templates And DTW

We compare the user's pitch shape to simple tone templates.

This section uses a small DTW implementation so you can see the core idea without extra libraries.

In [ ]:
def simple_dtw_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n, m = len(a), len(b)
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return float(dp[n, m] / (n + m))

def resample_curve(values, n=50):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return None
    x_old = np.linspace(0.0, 1.0, len(values))
    x_new = np.linspace(0.0, 1.0, n)
    return np.interp(x_new, x_old, values)

def normalize_f0_shape(f0_values, n=50):
    values = np.asarray(f0_values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[values > 0]
    if len(values) < 4:
        return None

    # Convert Hz to relative semitone-like units, then normalize shape.
    values = np.log2(values)
    values = resample_curve(values, n=n)
    values = values - np.nanmedian(values)
    scale = np.nanstd(values)
    if scale > 1e-6:
        values = values / scale
    return values

def tone_template(tone: str, n=50):
    x = np.linspace(0.0, 1.0, n)
    if tone == "1":
        y = np.zeros(n)
    elif tone == "2":
        y = -0.9 + 1.8 * x
    elif tone == "3":
        y = np.where(x < 0.55, 0.3 - 1.4 * (x / 0.55), -1.1 + 1.0 * ((x - 0.55) / 0.45))
    elif tone == "4":
        y = 1.0 - 2.0 * x
    else:
        y = np.zeros(n)

    y = y - np.median(y)
    scale = np.std(y)
    if scale > 1e-6:
        y = y / scale
    return y

def score_tone_from_f0(f0_segment, expected_tone: str):
    contour = normalize_f0_shape(f0_segment)
    if contour is None:
        return {
            "score": 0,
            "confidence": "low",
            "reason": "Too little voiced pitch was detected.",
            "contour": None,
            "template": tone_template(expected_tone).tolist(),
        }

    template = tone_template(expected_tone, n=len(contour))
    distance = simple_dtw_distance(contour, template)
    score = int(round(np.clip(100 * (1.0 - distance / 1.4), 0, 100)))

    return {
        "score": score,
        "confidence": "medium",
        "distance": distance,
        "contour": contour.tolist(),
        "template": template.tolist(),
    }

In [ ]:
def f0_values_in_window(f0_times, f0, start, end):
    mask = (f0_times >= start) & (f0_times <= end)
    return f0[mask]

tone_results = []

if audio is not None:
    for part, (start, end) in zip(syllable_parts, windows):
        segment_f0 = f0_values_in_window(f0_times, f0, start, end)
        result = score_tone_from_f0(segment_f0, part["tone"])
        tone_results.append({
            "pinyin": part["pinyin"],
            "tone": part["tone"],
            "start": start,
            "end": end,
            "tone_score": result["score"],
            "tone_result": result,
        })

tone_results

In [ ]:
def plot_tone_result(item):
    result = item["tone_result"]
    contour = result["contour"]
    template = result["template"]
    if contour is None:
        print(f"{item['pinyin']}: cannot plot because too little pitch was detected")
        return

    plt.figure(figsize=(5, 3))
    plt.plot(contour, label="user F0 shape", linewidth=2)
    plt.plot(template, label=f"tone {item['tone']} template", linewidth=2)
    plt.title(f"{item['pinyin']} tone score: {item['tone_score']}")
    plt.xlabel("Normalized time")
    plt.ylabel("Normalized pitch shape")
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.show()

for item in tone_results:
    plot_tone_result(item)

## 8. Optional: CTC Phoneme Recognition

This section downloads a Hugging Face model, so it may take time.

Set `RUN_CTC = True` when you are ready.

Beginner note: this model emits eSpeak-style phoneme labels, not Mandarin pinyin. Treat it as a learning scaffold. Later you should train a Mandarin pinyin/phone CTC model with WeNet, FunASR, ESPnet, or another ASR framework.

In [ ]:
RUN_CTC = False
CTC_MODEL_ID = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"

ctc_output = None

if RUN_CTC and audio is not None:
    import torch
    from transformers import AutoModelForCTC, AutoProcessor

    processor = AutoProcessor.from_pretrained(CTC_MODEL_ID)
    model = AutoModelForCTC.from_pretrained(CTC_MODEL_ID)
    model.eval()

    inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values).logits[0]
        probs = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(logits, dim=-1)

    decoded = processor.batch_decode(pred_ids[None, :])[0]
    ctc_output = {
        "decoded": decoded,
        "logits_shape": tuple(logits.shape),
        "vocab_size": logits.shape[-1],
    }
    print(json.dumps(ctc_output, indent=2, ensure_ascii=False))
else:
    print("CTC skipped. Set RUN_CTC = True after you have audio and dependencies installed.")

## 9. Simple Sequence Alignment

This is a small edit-distance aligner. It is useful for understanding alignment before you implement full CTC forced alignment.

Later, replace this with true CTC forced alignment over frame-level probabilities.

In [ ]:
def levenshtein_alignment(ref_tokens, hyp_tokens):
    n, m = len(ref_tokens), len(hyp_tokens)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    back = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i, 0] = i
        back[i][0] = "delete"
    for j in range(1, m + 1):
        dp[0, j] = j
        back[0][j] = "insert"

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub_cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            choices = [
                (dp[i - 1, j] + 1, "delete"),
                (dp[i, j - 1] + 1, "insert"),
                (dp[i - 1, j - 1] + sub_cost, "match" if sub_cost == 0 else "substitute"),
            ]
            dp[i, j], back[i][j] = min(choices, key=lambda x: x[0])

    aligned = []
    i, j = n, m
    while i > 0 or j > 0:
        op = back[i][j]
        if op in {"match", "substitute"}:
            aligned.append((op, ref_tokens[i - 1], hyp_tokens[j - 1]))
            i -= 1
            j -= 1
        elif op == "delete":
            aligned.append((op, ref_tokens[i - 1], None))
            i -= 1
        else:
            aligned.append((op, None, hyp_tokens[j - 1]))
            j -= 1

    aligned.reverse()
    return int(dp[n, m]), aligned

reference_units = []
for part in syllable_parts:
    if part["initial"]:
        reference_units.append(part["initial"])
    if part["final"]:
        reference_units.append(part["final"])

# Example hypothesis. Replace this with real CTC phoneme tokens later.
hypothesis_units = reference_units.copy()

distance, alignment = levenshtein_alignment(reference_units, hypothesis_units)
distance, alignment

## 10. Build App-Friendly Feedback JSON

This shape is close to what a backend API can return to a frontend.

In [ ]:
def tone_feedback_text(tone: str, score: int):
    if score >= 80:
        return "Tone shape looks close."
    if tone == "1":
        return "Try keeping the pitch flatter and steadier."
    if tone == "2":
        return "Try making the pitch rise more clearly."
    if tone == "3":
        return "Try making the pitch lower in the middle."
    if tone == "4":
        return "Try making the pitch fall more clearly."
    return "Try making the neutral tone shorter and lighter."

def build_feedback_json():
    syllables = []
    tone_by_pinyin = {x["pinyin"]: x for x in tone_results}

    for idx, part in enumerate(syllable_parts):
        tone_item = tone_by_pinyin.get(part["pinyin"])
        tone_score = int(tone_item["tone_score"]) if tone_item else None

        # Placeholder values until the CTC phone scorer is implemented.
        initial_score = None
        final_score = None

        available_scores = [s for s in [initial_score, final_score, tone_score] if s is not None]
        syllable_score = int(round(np.mean(available_scores))) if available_scores else None

        syllables.append({
            "index": idx,
            "char": TARGET_TEXT[idx] if idx < len(TARGET_TEXT) else "",
            "pinyin": part["pinyin"],
            "initial": part["initial"],
            "final": part["final"],
            "tone": part["tone"],
            "scores": {
                "initial": initial_score,
                "final": final_score,
                "tone": tone_score,
                "syllable": syllable_score,
            },
            "feedback": tone_feedback_text(part["tone"], tone_score) if tone_score is not None else "Add a recording to score this syllable.",
        })

    syllable_scores = [s["scores"]["syllable"] for s in syllables if s["scores"]["syllable"] is not None]
    overall_score = int(round(np.mean(syllable_scores))) if syllable_scores else None

    return {
        "text": TARGET_TEXT,
        "pinyin": pinyin_syllables,
        "overall_score": overall_score,
        "syllables": syllables,
        "notes": [
            "Tone scores use naive equal-length syllable windows in this notebook.",
            "Initial/final scores are placeholders until CTC forced alignment is added.",
        ],
    }

feedback = build_feedback_json()
print(json.dumps(feedback, indent=2, ensure_ascii=False))

## 11. What To Build Next

Recommended next steps:

1. Replace naive syllable windows with real alignment.
2. Use CTC forced alignment to align expected pinyin/phones to acoustic frames.
3. Add initial and final GOP-style scores.
4. Create a small FastAPI backend that returns the feedback JSON.
5. Build a frontend that displays syllable blocks and pitch curves.
6. Collect recordings and compare scores with human teacher labels.
7. Fine-tune a Mandarin pinyin/phone CTC model.

The most important technical upgrade is step 1: real alignment. Once syllable and phone timing are reliable, scoring becomes much more meaningful.